# Phase 7 prep aborted run — postmortem

Multi-study diagnostic for the 2026-04-19 aborted cloud run. Loads all 8
per-study `evaluation_log.jsonl` files from the experiment directory and
produces cross-hull comparisons for the win-rate anomaly.

Set `EXPERIMENT_DIR` via env var or edit below.

In [ ]:
import json
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EXPERIMENT_DIR = Path(os.environ.get(
    "EXPERIMENT_DIR",
    "../experiments/phase7-prep-aborted-2026-04-19",
)).resolve()

per_study_dirs = sorted((EXPERIMENT_DIR / "per-study").glob("*__early__tpe__seed_idx*"))
assert per_study_dirs, f"no per-study dirs found under {EXPERIMENT_DIR}"

rows = []
for d in per_study_dirs:
    hull = d.name.split("__")[0]
    log = d / "evaluation_log.jsonl"
    with open(log) as f:
        for line in f:
            r = json.loads(line)
            rows.append(r)

df_trials = pd.DataFrame({
    "hull":         [r["hull_id"]                 for r in rows],
    "trial":        [r["trial_number"]            for r in rows],
    "pruned":       [r["pruned"]                  for r in rows],
    "fitness":      [r["fitness"]                 for r in rows],
    "raw_fitness":  [r["raw_fitness"]             for r in rows],
    "n_opp":        [r["opponents_evaluated"]     for r in rows],
    "n_weapons":    [sum(1 for v in r["build"]["weapon_assignments"].values() if v) for r in rows],
    "n_hullmods":   [len(r["build"]["hullmods"]) for r in rows],
    "vents":        [r["build"]["flux_vents"]     for r in rows],
    "caps":         [r["build"]["flux_capacitors"] for r in rows],
})
print(f"Loaded {len(rows)} trials from {len(per_study_dirs)} studies")
print(df_trials.groupby("hull").agg(trials=("trial", "count"), pruned=("pruned", "sum"),
                                    fit_median=("fitness", "median"), fit_max=("fitness", "max")))

## 1. Per-hull win rate (all matchups)

In [ ]:
match_rows = []
for r in rows:
    for res in r["opponent_results"]:
        match_rows.append({
            "hull":     r["hull_id"],
            "trial":    r["trial_number"],
            "opponent": res["opponent"],
            "winner":   res["winner"],
            "duration": res["duration_seconds"],
            "hp_diff":  res["hp_differential"],
        })
df_match = pd.DataFrame(match_rows)

summary = df_match.groupby("hull").agg(
    matchups=("winner", "size"),
    wins=("winner", lambda s: (s == "PLAYER").sum()),
    losses=("winner", lambda s: (s == "ENEMY").sum()),
    timeouts=("winner", lambda s: (s == "TIMEOUT").sum()),
    dur_p50=("duration", "median"),
    dur_p95=("duration", lambda s: s.quantile(0.95)),
)
summary["win_pct"] = 100 * summary["wins"] / summary["matchups"]
summary["to_pct"] = 100 * summary["timeouts"] / summary["matchups"]
summary = summary.sort_values("win_pct")
print(summary.to_string(float_format="%.1f"))

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(summary.index, summary["win_pct"], color=plt.cm.RdYlGn(summary["win_pct"] / 100))
ax.axvline(50, color="gray", ls=":", alpha=0.4)
ax.set_xlabel("Player win %")
ax.set_title("Win rate by hull — the anomaly")
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

## 2. Outcome mix (W / L / T) per hull

In [ ]:
pivot = pd.crosstab(df_match["hull"], df_match["winner"], normalize="index").fillna(0) * 100
for col in ("PLAYER", "ENEMY", "TIMEOUT"):
    if col not in pivot.columns:
        pivot[col] = 0
pivot = pivot[["PLAYER", "ENEMY", "TIMEOUT"]].sort_values("PLAYER")

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind="barh", stacked=True, ax=ax,
           color=["#2ca02c", "#d62728", "#7f7f7f"])
ax.set_xlabel("% of matchups")
ax.set_title("Outcome mix per hull (PLAYER=green=optimizer wins)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()
print(pivot.to_string(float_format="%.1f"))

## 3. Losing to stock-same-hull variants

If the optimizer can't beat a stock variant of its OWN hull, that's
diagnostic. For wolf we look at opponents matching `wolf_*`, for lasher
`lasher_*`, etc.

In [ ]:
stock = df_match.apply(
    lambda r: r["opponent"].startswith(r["hull"] + "_"), axis=1,
)
vs_stock = df_match[stock]
same_hull = vs_stock.groupby("hull").agg(
    matchups=("winner", "size"),
    wins=("winner", lambda s: (s == "PLAYER").sum()),
    losses=("winner", lambda s: (s == "ENEMY").sum()),
    timeouts=("winner", lambda s: (s == "TIMEOUT").sum()),
)
same_hull["win_pct"] = 100 * same_hull["wins"] / same_hull["matchups"].clip(lower=1)
print("Optimizer vs stock variants of its OWN hull:")
print(same_hull.sort_values("win_pct").to_string(float_format="%.1f"))

## 4. Weapon slot fill-rate per hull

Hypothesis: failing hulls leave primary weapon slots empty more often.

In [ ]:
slot_fill = defaultdict(lambda: defaultdict(list))
for r in rows:
    for slot, weap in r["build"]["weapon_assignments"].items():
        slot_fill[r["hull_id"]][slot].append(weap is not None)

fill_rows = []
for hull in sorted(slot_fill):
    for slot in sorted(slot_fill[hull]):
        filled = slot_fill[hull][slot]
        fill_rows.append({
            "hull": hull, "slot": slot,
            "fill_pct": 100 * sum(filled) / len(filled),
            "n": len(filled),
        })
df_fill = pd.DataFrame(fill_rows)
print(df_fill.to_string(index=False, float_format="%.1f"))

fig, ax = plt.subplots(figsize=(10, 6))
for hull in sorted(slot_fill):
    sub = df_fill[df_fill["hull"] == hull]
    ax.plot(range(len(sub)), sub["fill_pct"], "o-", label=hull, alpha=0.7)
ax.set_ylabel("Slot fill %")
ax.set_xlabel("Slot index (sorted alphabetically per hull)")
ax.set_title("Per-slot fill rate — failing hulls should show empty primary slots")
ax.axhline(100, color="k", ls=":", alpha=0.3)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 5. Weapon fill rate by SLOT TYPE (ENERGY / BALLISTIC / MISSILE)

Working hulls lean on ballistic. Failing hulls mostly energy or missile.
Per-slot-type fill rate should reveal whether the optimizer is consistently
leaving ENERGY / MISSILE slots empty.

In [ ]:
import re
from pathlib import Path as _P

def read_hull(hull_id):
    raw = _P(f"../game/starsector/data/hulls/{hull_id}.ship").read_text()
    raw = re.sub(r"#.*", "", raw)
    raw = re.sub(r",(\s*[}\]])", r"\1", raw)
    return json.loads(raw)

slot_meta = {}
for hull_id in sorted({r["hull_id"] for r in rows}):
    try:
        d = read_hull(hull_id)
        slot_meta[hull_id] = {s["id"]: (s["type"], s["size"], s["mount"]) for s in d.get("weaponSlots", [])}
    except Exception as e:
        print(f"  skip {hull_id}: {e}")

type_fill_rows = []
for r in rows:
    hull = r["hull_id"]
    if hull not in slot_meta:
        continue
    for slot, weap in r["build"]["weapon_assignments"].items():
        meta = slot_meta[hull].get(slot)
        if meta is None:
            continue
        type_fill_rows.append({"hull": hull, "slot_type": meta[0], "slot_size": meta[1], "filled": weap is not None})
df_type = pd.DataFrame(type_fill_rows)

fill_by_type = df_type.groupby(["hull", "slot_type"]).agg(fill_pct=("filled", lambda s: 100 * s.mean()), n=("filled", "size"))
print(fill_by_type.to_string(float_format="%.1f"))

fig, ax = plt.subplots(figsize=(10, 6))
pivot_t = fill_by_type["fill_pct"].unstack().fillna(0)
pivot_t.plot(kind="bar", ax=ax)
ax.set_ylabel("Fill %")
ax.set_title("Weapon slot fill rate by SLOT TYPE per hull")
ax.set_ylim(0, 105)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 6. PRIMARY (MEDIUM / LARGE) slot fill rate

Small slots are secondary / PD. Large & medium slots are the ship's
primary firepower. If the optimizer leaves these empty, the ship has no
meaningful offense.

In [ ]:
primary = df_type[df_type["slot_size"].isin(["MEDIUM", "LARGE"])]
summary_primary = primary.groupby("hull").agg(
    n_primary_slots=("filled", "size"),
    fill_pct=("filled", lambda s: 100 * s.mean()),
).sort_values("fill_pct")
print("PRIMARY (MEDIUM+LARGE) slot fill per hull:")
print(summary_primary.to_string(float_format="%.1f"))

## 7. Correlation: primary-slot fill vs win rate

If the hypothesis holds, hulls with low primary-slot fill should have
low win rate.

In [ ]:
joined = summary_primary.join(summary[["win_pct", "to_pct"]])
print(joined.to_string(float_format="%.1f"))

fig, ax = plt.subplots(figsize=(8, 6))
for hull, row in joined.iterrows():
    ax.scatter(row["fill_pct"], row["win_pct"], s=80)
    ax.annotate(hull, (row["fill_pct"], row["win_pct"]), xytext=(5, 5), textcoords="offset points")
ax.set_xlabel("Primary (M+L) slot fill %")
ax.set_ylabel("Player win %")
ax.set_title("Primary-slot fill vs win rate")
ax.axhline(50, color="gray", ls=":", alpha=0.4)
plt.tight_layout()
plt.show()

if len(joined) >= 3:
    corr = joined[["fill_pct", "win_pct"]].corr().iloc[0, 1]
    print(f"Pearson correlation (fill_pct, win_pct) = {corr:.3f}")

## 8. "Best" build per failing hull

Pull the top-fitness build for each hull and print its loadout. For the
losing hulls, the best build should still reveal the structural issue.

In [ ]:
by_hull = defaultdict(list)
for r in rows:
    by_hull[r["hull_id"]].append(r)

for hull in sorted(by_hull):
    best = max(by_hull[hull], key=lambda r: r["fitness"])
    w = sum(1 for x in best["opponent_results"] if x["winner"] == "PLAYER")
    l = sum(1 for x in best["opponent_results"] if x["winner"] == "ENEMY")
    to = sum(1 for x in best["opponent_results"] if x["winner"] == "TIMEOUT")
    ws = {k: v for k, v in best["build"]["weapon_assignments"].items() if v}
    empty = [k for k, v in best["build"]["weapon_assignments"].items() if not v]
    print(f"=== {hull} === best fitness={best['fitness']:.3f} raw={best['raw_fitness']:.4f}  W{w}/L{l}/T{to}")
    print(f"  weapons (filled):  {ws}")
    print(f"  weapons (empty):   {empty}")
    print(f"  hullmods:          {best['build']['hullmods']}")
    print(f"  vents/caps:        {best['build']['flux_vents']}/{best['build']['flux_capacitors']}")
    print()

## 9. Duration distribution per hull

Timeout-clustered distributions (most matchups hit 300s) indicate
flux-stall or no-kill-throughput patterns.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for hull in sorted(df_match["hull"].unique()):
    dur = df_match[df_match["hull"] == hull]["duration"]
    ax.hist(dur, bins=60, alpha=0.4, label=f"{hull} (n={len(dur)})")
ax.set_xlabel("Matchup duration (s)")
ax.set_ylabel("Count")
ax.set_title("Matchup duration distribution per hull")
ax.axvline(300, color="red", ls="--", alpha=0.5, label="300s timeout")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## 10. Fitness distribution + warp passthrough reason

If raw_fitness is narrow (e.g. clustered near -1 for losing hulls) but
post-Box-Cox `fitness` is spread [0,1], the `shape_passthrough_reason`
tells us whether Box-Cox fired or fell through to min-max.

In [ ]:
passthru_by_hull = defaultdict(Counter)
for r in rows:
    passthru_by_hull[r["hull_id"]][r.get("shape_passthrough_reason")] += 1
for hull in sorted(passthru_by_hull):
    print(f"  {hull:12s} {dict(passthru_by_hull[hull])}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for hull in sorted(df_trials["hull"].unique()):
    sub = df_trials[df_trials["hull"] == hull]
    axes[0].hist(sub["raw_fitness"], bins=40, alpha=0.4, label=hull)
    axes[1].hist(sub["fitness"], bins=40, alpha=0.4, label=hull)
axes[0].set_xlabel("raw_fitness (TWFE α̂ / EB posterior)")
axes[0].set_title("Raw fitness per hull")
axes[1].set_xlabel("fitness (post-Box-Cox / min-max)")
axes[1].set_title("Warped fitness per hull")
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 11. Conclusion stub

What the notebook should localize:
- Is the anomaly driven by primary-slot fill (optimizer bug in scorer)?
- Or by raw-fitness compression at lower bound (all builds equally losing)?
- Or by specific opponent types that no build of the hull can counter?

Verdict goes in the experiment's `README.md` once analyzed.